# 08 — WWF LiDAR × Musanga V3_1
Explore the `gs://carbon-maps` DRC survey: 216 georeferenced plots, matching RGB mosaics, LAS clouds, CHM, DTM and DSM. This notebook is **read-only** on GCS and does not modify V3_1 model products. Run in Colab top to bottom. Large imagery is not downloaded automatically.

In [ ]:
# 00 — Install dependencies
%pip -q install google-cloud-storage geopandas shapely pyogrio rasterio folium earthengine-api geemap fsspec gcsfs

In [ ]:
# 01 — Authenticate Google Cloud, mount Drive and initialize Earth Engine
from google.colab import auth, drive
from pathlib import Path
import os
auth.authenticate_user()
drive.mount('/content/drive', force_remount=False)
EE_PROJECT = os.environ.get('EE_PROJECT', '').strip()  # specify your registered GEE Cloud project if necessary
import ee
try:
    if EE_PROJECT: ee.Initialize(project=EE_PROJECT)
    else: ee.Initialize()
    print('Earth Engine ready')
except Exception as e:
    print('GEE not initialized (not required for LiDAR exploration):', repr(e))
    print('Set EE_PROJECT and rerun this cell to enable satellite comparisons.')
MUSANGA_ROOT = Path('/content/drive/MyDrive/Colab Notebooks/Musanga')
WORK_DIR = MUSANGA_ROOT/'V3_1'/'NB08_WWF_Lidar'
WORK_DIR.mkdir(parents=True,exist_ok=True)
print('Outputs:',WORK_DIR)

In [ ]:
# 02 — Bucket inventory (metadata only)
from google.cloud import storage
import pandas as pd, re
from pathlib import PurePosixPath
BUCKET='carbon-maps'
client=storage.Client()
rows=[]
for b in client.list_blobs(BUCKET):
    if b.name.endswith('/'): continue
    m=re.search(r'(UTM\d{1,2}[NS])[_/-]?(?:Plot|plot)[_-]?(\d+)',b.name,re.I)
    plot_id=f'{m.group(1).upper()}_Plot{int(m.group(2))}' if m else None
    rows.append(dict(path=b.name,extension=PurePosixPath(b.name).suffix.lower(),size_GB=(b.size or 0)/1e9,plot_id=plot_id,updated=str(b.updated)))
df=pd.DataFrame(rows)
df.to_csv(WORK_DIR/'carbon_maps_inventory.csv',index=False)
print('Files:',len(df),'GB:',round(df.size_GB.sum(),1))
display(df.groupby('extension').agg(files=('path','size'),GB=('size_GB','sum')).sort_values('GB',ascending=False).head(20))

In [ ]:
# 03 — Match WWF LiDAR products to plot IDs
patterns={
 'RGB':r'^001_Lidar_Rasters/RGB_mosaic/.*_RGB\.tif$',
 'CHM':r'^001_Lidar_Rasters/Lidar_2meter/CHM/.*\.tif$',
 'DTM':r'^001_Lidar_Rasters/Lidar_2meter/DTM/.*\.tif$',
 'DSM':r'^001_Lidar_Rasters/Lidar_2meter/DSM/.*\.tif$',
 'LAS':r'^00-raw-data/Plots/.*\.las$',
 'ECW':r'^00-raw-data/Plots/.*\.ecw$'}
products=pd.DataFrame({'plot_id':sorted(df.plot_id.dropna().unique())})
for kind,pattern in patterns.items():
    counts=df.loc[df.path.str.contains(pattern,case=False,regex=True,na=False)].groupby('plot_id').size()
    products[kind+'_n']=products.plot_id.map(counts).fillna(0).astype(int)
display(products.head(20))
print('Plots with RGB:',(products.RGB_n>0).sum())
products.to_csv(WORK_DIR/'wwf_plot_product_inventory.csv',index=False)

In [ ]:
# 04 — Download only small plot GIS sidecars; do not read broken Ferry_Lines geometry
import tempfile
import geopandas as gpd
from pathlib import Path
prefix='003_GIS_Layers/Lidar_Plots/'
local=Path(tempfile.mkdtemp(prefix='wwf_gis_'))
for b in client.list_blobs(BUCKET,prefix=prefix):
    if b.name.lower().endswith(('.shp','.shx','.dbf','.prj','.cpg','.qpj')):
        b.download_to_filename(str(local/Path(b.name).name))
shps=list(local.glob('*.shp'))
assert shps, 'No plot shapefile found'
plots=gpd.read_file(shps[0]).to_crs('EPSG:4326')
plots=plots[plots.geometry.notna() & ~plots.geometry.is_empty].copy()
plots['geometry']=plots.geometry.make_valid()
plots['plot_id']=plots.Name.astype(str).str.strip()
plots['utm_zone']=plots.plot_id.str.extract(r'^(UTM\d{1,2}[NS])',expand=False)
plots=plots.merge(products,on='plot_id',how='left',validate='one_to_one')
print('Features:',len(plots),'Unique plots:',plots.plot_id.nunique())
display(plots.groupby('utm_zone').agg(plots=('plot_id','size'),rgb=('RGB_n',lambda x:(x>0).sum())))
plots.to_file(WORK_DIR/'wwf_lidar_plots.gpkg',driver='GPKG')

In [ ]:
# 05 — Robust interactive map (NO OpenStreetMap requests)
# The previous OSM basemap returned HTTP 403 within Colab.
# Esri satellite is optional; the default offline map always works.
import folium
from folium.plugins import Fullscreen, MiniMap
from IPython.display import display
from html import escape
import pandas as pd

# tiles=None ensures ZERO requests to restricted OSM tile servers.
m = folium.Map(location=[-2.5, 23.5], zoom_start=6, tiles=None,
               control_scale=True, prefer_canvas=True)

# Reliable fallback: blank geographic background (no external tile dependency).
folium.TileLayer(
    tiles='data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAQAAAC1HAwCAAAAC0lEQVR42mP8/x8AAusB9Y9UxQAAAABJRU5ErkJggg==',
    attr='Local blank geographic background', name='Offline (always works)',
    overlay=False, show=True, control=True, max_zoom=19
).add_to(m)

# Optional imagery; switch it on only if your Colab browser can access Esri.
folium.TileLayer(
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
    attr='Tiles © Esri', name='Esri World Imagery (optional)',
    overlay=False, show=False, control=True, max_zoom=19
).add_to(m)

colors={'UTM33S':'#9467bd','UTM34S':'#e3771c','UTM34N':'#2ca02c',
        'UTM35S':'#d62728','UTM35N':'#1f77b4','UTM36N':'#8c564b'}

# Plot geometry gives survey context even if ALL basemap servers fail.
foot=folium.FeatureGroup(name='Survey footprints',show=True)
folium.GeoJson(
    plots[['plot_id','geometry']].to_json(),
    style_function=lambda f: {'color':'#555555','weight':1.5,'fillOpacity':0.08},
    tooltip=folium.GeoJsonTooltip(fields=['plot_id'])
).add_to(foot)
foot.add_to(m)

for zone,group in plots.groupby('utm_zone'):
    layer=folium.FeatureGroup(name=f'{zone}: {len(group)} plots',show=True)
    for _,r in group.iterrows():
        pt=r.geometry.representative_point()
        counts={k:int(r[k+'_n']) if pd.notna(r[k+'_n']) else 0
                for k in ('RGB','LAS','CHM','DTM','DSM')}
        label=escape(str(r.plot_id))
        html='<b>'+label+'</b><br>'+'<br>'.join(f'{k}: {v}' for k,v in counts.items())
        col=colors.get(zone,'gray')
        folium.CircleMarker(
            [pt.y,pt.x],radius=5,color=col,weight=2,fill=True,
            fill_color=col if counts['RGB'] else '#ffffff',fill_opacity=.9,
            tooltip=label,popup=folium.Popup(html,max_width=300)
        ).add_to(layer)
    layer.add_to(m)

# Geographic graticule: geographic reference even with the offline background.
graticule=folium.FeatureGroup(name='Latitude / longitude grid',show=True)
for latitude in range(-12,7,2):
    folium.PolyLine([[latitude,12],[latitude,32]],color='#a0a0a0',
                    weight=.6,opacity=.4,tooltip=f'{latitude}° latitude').add_to(graticule)
for longitude in range(12,33,2):
    folium.PolyLine([[-13,longitude],[7,longitude]],color='#a0a0a0',
                    weight=.6,opacity=.4,tooltip=f'{longitude}° longitude').add_to(graticule)
graticule.add_to(m)

w,s,e,n=plots.total_bounds
m.fit_bounds([[s,w],[n,e]])
Fullscreen().add_to(m)
folium.LayerControl(collapsed=False).add_to(m)

output=WORK_DIR/'wwf_lidar_survey_map.html'
m.save(str(output))
print('Offline-capable map:',output)
print('To see satellite imagery: select Esri World Imagery in the layer control.')
display(m)

## 05A — Overlay approved Musanga UAV ROIs
Uses the V3_1 approved input manifest and `ROI.shp`, with site assignment from mosaic footprint intersections. Run after Block 05.

In [ ]:
# 05A — Overlay the real Musanga V3_1 UAV ROI polygons on the WWF map
# Source of truth: V3_1 audited manifest and its ROI.shp (no guessed coordinates).
from pathlib import Path
from shapely.geometry import box
import geopandas as gpd
import rasterio
import folium
import pandas as pd
from IPython.display import display
import json
from html import escape

drive_root=Path('/content/drive/MyDrive')
final_root=next((q for q in [
    drive_root/'Colab Notebooks/Musanga/Final',
    drive_root/'Musanga/Final'] if q.is_dir()), None)
assert final_root, 'Musanga/Final not found. Mount Drive and check V3_1 paths.'
manifest_file=final_root/'Outputs/V3_1/00_input_audit/00_input_manifest.json'
assert manifest_file.exists(), (
    'Approved V3_1 input manifest missing: '+str(manifest_file)
    +'. Run V3_1 Notebook 00 / Block 10 first.')
manifest=json.loads(manifest_file.read_text())
roi_file=Path(manifest['roi'])
assert roi_file.is_file(), f'ROI shapefile missing: {roi_file}'
uav_roi=gpd.read_file(roi_file)
assert uav_roi.crs is not None, 'ROI.shp has no CRS; refusing to guess.'
uav_roi=uav_roi[uav_roi.geometry.notna() & ~uav_roi.geometry.is_empty].copy()
uav_roi.geometry=uav_roi.geometry.make_valid()
uav_roi=uav_roi[uav_roi.geometry.notna() & ~uav_roi.geometry.is_empty].copy()
assert len(uav_roi), 'No valid UAV ROI geometries.'

# Assign ROI polygons to a site by physical overlap with real mosaic footprints.
# Intersections are measured in the ROI's native metric CRS, not lat/lon degrees.
mosaic_records=[]
for entry in manifest['mosaics']:
    image_file=Path(entry['path'])
    if not image_file.exists():
        print('Mosaic missing; cannot assign this footprint:',image_file)
        continue
    with rasterio.open(image_file) as ds:
        assert ds.crs is not None, f'No CRS: {image_file}'
        g=gpd.GeoSeries([box(*ds.bounds)],crs=ds.crs).to_crs(uav_roi.crs).iloc[0]
        mosaic_records.append(dict(site=str(entry['site']),file=str(entry['file']),geometry=g))
assert mosaic_records, 'No readable UAV mosaics for ROI attribution.'

# Aggregate overlapping mosaics by forest site, avoiding duplicate MOS geometry.
from shapely.ops import unary_union
site_footprints={
    site:unary_union([x['geometry'] for x in mosaic_records if x['site']==site])
    for site in sorted(set(x['site'] for x in mosaic_records))}
roi_rows=[]
for index,row in uav_roi.iterrows():
    g=row.geometry
    overlaps={site:float(g.intersection(foot).area)
              for site,foot in site_footprints.items()}
    best=max(overlaps,key=overlaps.get)
    if overlaps[best]<=0:
        best='Unassigned'
    roi_rows.append(best)
uav_roi['site']=roi_rows
uav_roi['roi_row']=[int(i) for i in range(len(uav_roi))]
print('UAV ROI features:',len(uav_roi))
print('Site assignment:',uav_roi.site.value_counts().to_dict())
if (uav_roi.site=='Unassigned').any():
    print('WARNING: Unassigned ROI polygons are shown separately, not silently removed.')

roi_wgs=uav_roi.to_crs('EPSG:4326')
roi_colors={'Baego':'#ef4444','MOS':'#f59e0b','Yangambi':'#10b981',
            'Yoko':'#ec4899','Unassigned':'#111827'}
for site, group in roi_wgs.groupby('site'):
    layer=folium.FeatureGroup(name=f'UAV ROI — {site} ({len(group)})',show=True)
    color=roi_colors.get(site,'#111827')
    folium.GeoJson(
        group[['site','roi_row','geometry']].to_json(),
        style_function=lambda feat,c=color:{
            'color':c,'weight':3.5,'fillColor':c,'fillOpacity':.16},
        highlight_function=lambda feat:{'weight':5,'fillOpacity':.3},
        tooltip=folium.GeoJsonTooltip(fields=['site','roi_row'],
            aliases=['Musanga site','ROI row'])
    ).add_to(layer)
    # Distinct labels at true ROI interiors
    for _,rr in group.iterrows():
        pt=rr.geometry.representative_point()
        folium.Marker([pt.y,pt.x],
            icon=folium.DivIcon(html=(
                '<div style="font: bold 11px sans-serif;white-space:nowrap;'
                'text-shadow:1px 1px 2px white,-1px -1px 2px white;color:'
                +color+';">'+escape(site)+'</div>'))
        ).add_to(layer)
    layer.add_to(m)

# Drop old layer controls before adding the revised one.
for key, child in list(m._children.items()):
    if isinstance(child,folium.map.LayerControl):
        del m._children[key]
folium.LayerControl(collapsed=False).add_to(m)

# Plot boundaries remain visible against the complete WWF footprint.
allbounds=pd.concat([
    pd.Series([plots.total_bounds[0],roi_wgs.total_bounds[0]]),
    pd.Series([plots.total_bounds[1],roi_wgs.total_bounds[1]]),
    pd.Series([plots.total_bounds[2],roi_wgs.total_bounds[2]]),
    pd.Series([plots.total_bounds[3],roi_wgs.total_bounds[3]])
],axis=1)
m.fit_bounds([[min(plots.total_bounds[1],roi_wgs.total_bounds[1]),
               min(plots.total_bounds[0],roi_wgs.total_bounds[0])],
              [max(plots.total_bounds[3],roi_wgs.total_bounds[3]),
               max(plots.total_bounds[2],roi_wgs.total_bounds[2])]])

uav_roi.to_file(WORK_DIR/'musanga_v3_1_roi_wwf_overlay.gpkg',driver='GPKG')
map_path=WORK_DIR/'wwf_lidar_plus_musanga_uav_roi.html'
m.save(str(map_path))
print('Saved map:',map_path)
print('Saved georeferenced ROI:',WORK_DIR/'musanga_v3_1_roi_wwf_overlay.gpkg')
display(m)

In [ ]:
# 06 — Compute nearest LiDAR plots once verified Musanga UAV ROI coordinates are provided
# Enter coordinates from V3_1 ROI shapefiles; do not substitute town centres.
MUSANGA_SITES={}  # e.g. {'Yangambi': (lat,lon), 'Yoko': (lat,lon)}
from pyproj import Geod
geod=Geod(ellps='WGS84')
if MUSANGA_SITES:
    table=[]
    for name,(lat,lon) in MUSANGA_SITES.items():
        for _,r in plots.iterrows():
            c=r.geometry.representative_point()
            _,_,dist=geod.inv(lon,lat,c.x,c.y)
            table.append(dict(site=name,plot_id=r.plot_id,distance_km=dist/1000,RGB_n=r.RGB_n,LAS_n=r.LAS_n,CHM_n=r.CHM_n))
    near=pd.DataFrame(table).sort_values('distance_km').groupby('site').head(10)
    display(near)
    near.to_csv(WORK_DIR/'nearest_musanga_wwf_plots.csv',index=False)
else: print('Enter georeferenced UAV ROI coordinates to calculate nearest plots.')

In [ ]:
# 07 — Inspect one RGB GeoTIFF remotely without transferring the entire mosaic
import rasterio
sample=df[df.path.str.contains(patterns['RGB'],case=False,regex=True,na=False)].iloc[0]
uri='/vsigs/'+BUCKET+'/'+sample.path
print('Raster:',sample.path,'Size (GB):',round(sample.size_GB,3))
try:
    with rasterio.open(uri) as src:
        print('CRS:',src.crs,'Size:',src.width,src.height,'Bands:',src.count)
        print('Pixel size:',src.res,'Bounds:',src.bounds,'Overviews:',src.overviews(1))
except Exception as exc:
    print('GDAL virtual GCS read unavailable:',repr(exc))
    print('Try a selected small CHM with gsutil cp after checking object size.')

## Follow-up experiments
1. Compare RGB and LiDAR acquisition dates, CRS, pixel resolutions, and horizontal alignment.
2. Overlay actual V3_1 UAV ROI geometries to identify survey overlap.
3. Inspect small matching RGB and CHM windows before selecting LAS subsets.
4. Explore canopy heights, gaps, and crown metrics, and eventually link them to Musanga maps and Sentinel-2/GEE. Avoid treating historical and contemporary observations as simultaneous.